<a href="https://colab.research.google.com/github/lavanya-001/CSA6102-DIGITAL-FORENSICS-LAB/blob/main/DFIR_EXPERIMENTS_37.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from datetime import datetime
from collections import Counter

LOG_FMT = "%Y-%m-%d %H:%M:%S"


def build_baseline_ips(logs):
    """
    Return a dictionary:
    {user: most_common_ip}
    based on all log activity.
    """

    by_user = {}

    for entry in logs:
        by_user.setdefault(entry["user"], []).append(entry["ip"])

    return {
        user: Counter(ips).most_common(1)[0][0]
        for user, ips in by_user.items()
    }


def flag_anomalous_downloads(logs, business_start=8, business_end=20):
    """
    Flag download events that:
    - Come from an IP different from the user's baseline IP.
    - Occur outside business hours.
    """

    baseline = build_baseline_ips(logs)

    flagged = []

    for entry in logs:

        if entry["action"] != "download":
            continue

        ts = datetime.strptime(entry["timestamp"], LOG_FMT)

        reasons = []

        if entry["ip"] != baseline.get(entry["user"]):
            reasons.append("IP differs from user baseline")

        if not (business_start <= ts.hour < business_end):
            reasons.append("Outside business hours")

        if reasons:
            flagged.append({
                **entry,
                "reasons": reasons
            })

    return flagged

In [2]:
logs = [
    {
        "user": "alice",
        "ip": "192.168.1.10",
        "action": "login",
        "timestamp": "2026-08-04 09:00:00"
    },
    {
        "user": "alice",
        "ip": "192.168.1.10",
        "action": "download",
        "timestamp": "2026-08-04 10:00:00"
    },
    {
        "user": "alice",
        "ip": "10.0.0.5",
        "action": "download",
        "timestamp": "2026-08-04 22:30:00"
    },
    {
        "user": "bob",
        "ip": "172.16.1.20",
        "action": "download",
        "timestamp": "2026-08-04 11:00:00"
    }
]

print(build_baseline_ips(logs))
print(flag_anomalous_downloads(logs))

{'alice': '192.168.1.10', 'bob': '172.16.1.20'}
[{'user': 'alice', 'ip': '10.0.0.5', 'action': 'download', 'timestamp': '2026-08-04 22:30:00', 'reasons': ['IP differs from user baseline', 'Outside business hours']}]
